# Human-in-the-Loop

*Notebook 24*

Pause agent execution at critical decision points and require human approval before continuing.
<br>
<br>

**Topics:**

- When to pause for human approval

- Marking tools with `needs_approval=True`

- The pause → inspect → approve/reject → resume flow

- Threshold-based escalation — auto-approve low-risk, escalate high-risk

---

## 🔧 Setup

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

from agents import Agent, Runner, function_tool

# Notebook-specific imports
import json

MODEL = "gpt-5-mini"


print("✅ Ready!")

---

## 🎯 The Problem

Some agent actions are reversible — getting the weather, looking up a product. For high-stakes actions, you want a human to review and approve before the tool executes.

---

## 🔄 Part 1: How HITL Works

The tool never executes until you explicitly approve it. The approval flow has four steps:
1. Runner.run() → agent decides to call a tool marked needs_approval=True

2. SDK pauses — tool does NOT execute yet. `result.interruptions` contains the pending approvals.

3. You convert the run result with `state = result.to_state()`, then call `state.approve(interruption)` or `state.reject(interruption)`.

4. `Runner.run(agent, state)` resumes from the paused point. Approved → tool executes; rejected → agent adapts.

---

## ✉️ Part 2: Basic Approval Flow

We'll build an email agent where sending always requires human approval. This matters because approval now happens in the SDK — the tool cannot run just because the agent ignored its instructions.

In [ ]:
@function_tool
def draft_email(to: str, subject: str, body: str) -> str:
    """Draft an email (no approval needed — just composing)."""
    return f"Draft ready: To={to}, Subject={subject}, Body={body[:50]}..."


@function_tool(needs_approval=True)
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to the recipient. Requires human approval before sending."""
    # In production this would call an email API
    return f"✅ Email sent to {to}: {subject}"


instructions = (
    "Help users compose and send emails.\n"
    "Use draft_email to prepare the message, then send_email to deliver it."
)

email_agent = Agent(
    name="EmailAssistant",
    instructions=instructions,
    model=MODEL,
    tools=[draft_email, send_email]
)

# --------------------------------------------------------------
print("✅ Email agent ready")
print("   send_email requires approval — draft_email does not")

#### Step 1: Run Until Paused

In [ ]:
print("📧 EMAIL APPROVAL DEMO")
print("=" * 60)

result = await Runner.run(email_agent, input="Send an email to alex@example.com with subject 'Meeting Tomorrow' saying we'll meet at 2pm.")

if result.interruptions:
    print(f"⏸️  Run paused — {len(result.interruptions)} approval(s) pending\n")
    for interruption in result.interruptions:
        print(f"🔔 Tool: {interruption.raw_item.name}")
        print(f"    Args: {interruption.raw_item.arguments}")
        # Note: arguments is a JSON string — use json.loads() to inspect specific fields
else:
    print("Run completed without interruptions")
    print(result.final_output)

Each interruption wraps a `raw_item` — that's the pending tool call, with `.name` and `.arguments` (as a JSON string) available for inspection.

#### Step 2: Approve and Resume

In [ ]:
if result.interruptions:
    # Capture the resumable state
    state = result.to_state()

    # Approve all pending interruptions
    for interruption in state.get_interruptions():
        print(f"✅ Approving: {interruption.raw_item.name}")
        state.approve(interruption)

    # Resume the run

    resumed_result = await Runner.run(email_agent, state)

    print(f"\n📨 Run resumed and completed")
    print(f"Response: {resumed_result.final_output}")
    print("=" * 60)

### 💡 Why This Works

`needs_approval=True` marks the tool as requiring a human decision. The SDK pauses the run automatically and surfaces the pending call in `result.interruptions`. The tool never runs until you call `state.approve()`.

This is the reusable pattern: run until interrupted, convert to `state`, approve or reject each interruption, then resume with that same state.

---

## ❌ Part 3: Rejecting a Tool Call

When you reject, the agent receives a rejection message and adapts — it doesn't crash.

In [ ]:
print("🚫 REJECTION DEMO")
print("=" * 60)

result = await Runner.run(email_agent, input="Send an urgent email to all-company@example.com saying the servers are down.")

if result.interruptions:
    state = result.to_state()

    for interruption in state.get_interruptions():
        print(f"🔔 Tool wants to run: {interruption.raw_item.name}")
        print(f"    Args: {interruption.raw_item.arguments}\n")

        # Reject — provide a reason the agent can use
        print("🚫 Rejecting — all-company emails require manager sign-off")
        state.reject(
            interruption,
            message="All-company emails require manager approval. Please save as draft instead."
        )

    # Resume — agent adapts to the rejection

    resumed_result = await Runner.run(email_agent, state)

    print(f"\nAgent response after rejection:")
    print(resumed_result.final_output)
    print("=" * 60)

Rejection is useful because the agent receives context it can use to adapt instead of failing or retrying the same forbidden action.

---

## 🎚️ Part 4: Threshold-Based Escalation

Not every action needs the same level of oversight. A common pattern: auto-approve actions below a risk threshold, escalate to a human for high-risk ones, based on the tool's parameters. The threshold here represents a confidence/risk level — low-value refunds are low-risk (auto-approve), high-value refunds require human review.

The SDK does not choose this threshold for you — you write the rule in Python, such as `if amount > 50: # require approval`.

In [ ]:
@function_tool(needs_approval=True)
def process_refund(order_id: str, amount: float, reason: str) -> str:
    """Process a refund for a customer order."""
    return f"✅ Refund of ${amount:.2f} processed for order {order_id}"


@function_tool
def lookup_order(order_id: str) -> str:
    """Look up order details by ID."""
    # Simulated order data
    orders = {
        "ORD-001": "$15.99 — Phone case, purchased 3 days ago",
        "ORD-002": "$349.00 — Headphones, purchased 1 week ago",
        "ORD-003": "$8.50 — Phone screen protector, purchased today"
    }
    return orders.get(order_id, f"Order {order_id} not found")


support_agent = Agent(
    name="SupportAgent",
    instructions="You are a customer support agent. Help customers with refunds and order issues.",
    model=MODEL,
    tools=[lookup_order, process_refund]
)

# --------------------------------------------------------------
print("✅ Support agent ready")

#### Threshold-Based Approval Logic

In [ ]:
async def handle_refund_request(customer_message, auto_approve_threshold=50.0):
    """Handle refund with threshold-based approval."""

    print(f"📞 Customer: {customer_message}")
    print("=" * 60)

    result = await Runner.run(support_agent, input=customer_message)

    # Use a while loop because the agent may call another approval-required tool after resuming
    # — e.g., agent looks up the order, then asks to refund, then asks to send a confirmation email
    while result.interruptions:
        state = result.to_state()

        for interruption in state.get_interruptions():
            tool_name = interruption.raw_item.name
            # arguments comes back as a JSON string — parse it to read individual fields
            args = json.loads(interruption.raw_item.arguments)

            if tool_name == "process_refund":
                amount = args.get("amount", 0)

                if amount <= auto_approve_threshold:
                    print(f"✅ Auto-approved: ${amount:.2f} refund (below ${auto_approve_threshold} threshold)")
                    state.approve(interruption)
                else:
                    # Simulate human review
                    print(f"⚠️  Manual review required: ${amount:.2f} refund")
                    print(f"    Order: {args.get('order_id')}")
                    print(f"    Reason: {args.get('reason')}")
                    # In production, prompt a human via UI or messaging system
                    # In your own app, this is the line you'd replace with a real approval source
                    # — a button click in a UI, a Slack reply, or a CLI input() prompt.
                    decision = "yes"  # Simulated approval for demo purposes

                    if decision in ["yes", "y"]:
                        print("    ✅ Approved by human")
                        state.approve(interruption)
                    else:
                        print("    🚫 Rejected by human")
                        state.reject(interruption, message="Refund requires additional verification. Please contact billing.")
            else:
                state.approve(interruption)

        result = await Runner.run(support_agent, state)

    print(f"\n💬 Agent: {result.final_output}")
    return result


# --------------------------------------------------------------
print("✅ handle_refund_request() ready")

### 💡 Why This Works

The same `needs_approval=True` flag serves both paths — auto-approve and human escalation. The SDK doesn't distinguish between them; it always pauses and surfaces the interruption. Your code inspects the parameters and decides which path to take.

⚠️ **Security note:** Refunds and financial write operations are irreversible. Always require explicit human approval for high-value actions — never rely on the agent's judgment alone.

#### Test: Small Refund (Auto-Approved)

In [ ]:
# Small refund — auto-approved
await handle_refund_request(
    "I'd like a refund for order ORD-001. The phone case broke after one day.",
    auto_approve_threshold=50.0
)

#### Test: Large Refund (Requires Human Review)

The approval decision is hardcoded to `"yes"` for this demo. In a real implementation, this would prompt the user via a UI or `input()` call before proceeding.

In [ ]:
# Large refund — requires manual review
await handle_refund_request(
    "I want to return order ORD-002. The headphones never worked properly.",
    auto_approve_threshold=50.0
)

---

## 💪 Practice Exercises

### Exercise 1: File Operations Agent

*Covers: approval callbacks — read vs. delete permission tiers*

Build an agent with read (no approval) and delete (requires approval) file tools.

⚠️ **Security note:** Filesystem delete tools are high-risk side-effect tools. In real systems, always require strict scoping and explicit approval before allowing write or delete actions.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: File Operations Agent
# --------------------------------------------------------------
# Objective: Read is safe — delete requires approval.

# TODO 1: Create a read_file tool (no approval needed)
#           Returns simulated file content for a given filename

# TODO 2: Create a delete_file tool with needs_approval=True
#           Returns a deletion confirmation message

# TODO 3: Create a FileAgent with both tools

# TODO 4: Run with: "Read config.json and then delete temp.log"
#           Print the interruption details before approving
#           Approve and resume, print the final output

# --- Write your code below this line ---

### Exercise 2: Budget-Aware Purchase Agent

*Covers: approval callbacks — threshold-based auto-approval*

Build an agent that auto-approves small purchases and requires approval for large ones.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 2: Budget-Aware Purchase Agent
# --------------------------------------------------------------
# Objective: Auto-approve purchases under $25, require approval above.

# TODO 1: Create a make_purchase tool with needs_approval=True
#           Parameters: item (str), price (float)
#           Returns a purchase confirmation string

# TODO 2: Create a PurchaseAgent with this tool

# TODO 3: Write a purchase_handler() function that:
#           - Runs the agent
#           - If interrupted, checks the price from the tool arguments
#           - Auto-approves if price <= 25
#           - Prints "Manual approval required" and approves if price > 25
#           - Resumes and prints final_output

# TODO 4: Test with:
#           - "Buy a pen for $3.99"
#           - "Buy a standing desk for $450"

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**The Approval Flow:**

- Mark tools with `@function_tool(needs_approval=True)`

- `Runner.run()` pauses — inspect `result.interruptions`

- Call `state.approve(interruption)` or `state.reject(interruption, message="...")`

- Resume with `Runner.run(agent, state)`

- The tool never runs until you call `state.approve()` — rejection gives the agent feedback to adapt
<br>
<br>

**Threshold-Based Escalation:**

- Auto-approve low-risk actions programmatically

- Escalate to humans only when parameters exceed safe thresholds

- The rejection message gives the agent context to adapt gracefully

---

## 📍 Next Step

**Notebook 25: Tracing & Observability**  

Learn how to read traces in the OpenAI dashboard to inspect tool calls, handoffs, and debug bad runs.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-24-human-in-the-loop)